# 07 — Real-world Flask Analysis

Notebooks 01–06 introduced the forward pipeline on toy fixtures. This notebook points COGANT at a **realistic Flask-style web application** that ships under `examples/real_world/flask_app/` and asks the same questions at scale:

1. What does the `ProgramGraph` look like for a module that uses decorators, middleware, a service container, and error handlers?
2. Which framework patterns become OBSERVATION, ACTION, or CONSTRAINT in the translated GNN?
3. How is the role distribution different from a numerically-oriented fixture like `calculator`?

**Run from the `cogant/` directory:**
```bash
uv run jupyter nbconvert --to notebook --execute docs/notebooks/07_real_world_flask.ipynb
```

## 1. Locate the Flask fixture

The fixture is a deliberately framework-free Flask lookalike — it reimplements routing, middleware, request/response objects, and error handling in a few hundred lines so you can run COGANT on it without needing Flask installed. This keeps the notebook hermetic across environments.

In [1]:
from __future__ import annotations

from pathlib import Path

import cogant
from cogant.api.session import Session

FIXTURE = Path("examples/real_world/flask_app/").resolve()
if not FIXTURE.exists():
    raise FileNotFoundError(
        f"Flask fixture not found at {FIXTURE}. Run this notebook from the cogant/ directory."
    )

print(f"COGANT version : {cogant.__version__}")
print(f"Fixture        : {FIXTURE}")
print(f"Python files   : {sorted(p.name for p in FIXTURE.glob('*.py'))}")

COGANT version : 0.5.0
Fixture        : /Users/4d/Documents/GitHub/template/projects_in_progress/cogant/cogant/examples/real_world/flask_app
Python files   : ['__init__.py', 'app.py', 'config.py', 'models.py', 'services.py', 'utils.py']


## 2. Build the program graph

`Session.build_graph()` runs ingest → static analysis → normalize → graph. For a real-world module this surfaces classes (`Application`, `Route`, `Middleware`, …), methods, module-level helpers, and the edges between them.

In [2]:
session = Session(repo_path=FIXTURE)
graph = session.build_graph()

nodes = graph.get("nodes", {})
edges = graph.get("edges", {})

print(f"Graph type  : {graph.get('type')}")
print(f"Nodes       : {len(nodes)}")
print(f"Edges       : {len(edges)}")
print(f"Languages   : {graph.get('metadata', {}).get('languages')}")

Graph type  : program_graph
Nodes       : 98
Edges       : 154
Languages   : ['python']


## 3. Role / kind distribution

Count node kinds so you can eyeball the structural shape of the Flask app. A web module typically has many methods (route handlers), a handful of classes (`Application`, `Middleware`, `Route`), and a non-trivial number of module-level helpers.

In [3]:
from collections import Counter

kind_counts = Counter(n.get("kind", "unknown") for n in nodes.values())
print("Kind distribution:")
for kind, count in kind_counts.most_common():
    print(f"  {kind:<12} {count}")

Kind distribution:
  method       57
  class        25
  function     10
  module       6


## 4. Translate to GNN sections

`translate_to_gnn()` runs the built-in rule engine and returns a GNN-oriented summary keyed by `mapping_ids`, `node_features`, and `edge_indices`. Under the hood, the session bundle also holds a dict of `SemanticMapping` objects — those are the richer records we'll use to count roles and trace them back to the original graph nodes.

In [4]:
gnn = session.translate_to_gnn()
print(f"GNN type       : {gnn.get('type')}")
print(f"Top-level keys : {sorted(gnn.keys())}")
print(f"Mapping count  : {gnn.get('mapping_count')}")

# Pull the full SemanticMapping dict from the session's internal
# bundle. Each value has a `.kind` (MappingKind enum) plus a list of
# graph node ids the mapping covers.
semantic_mappings: dict = session._bundle.artifacts.get("_semantic_mappings", {})  # noqa: SLF001
print(f"SemanticMappings: {len(semantic_mappings)}")

GNN type       : gnn_model
Top-level keys : ['edge_indices', 'embeddings', 'mapping_count', 'mapping_ids', 'node_features', 'type']
Mapping count  : 68
SemanticMappings: 68


## 5. Role distribution over mapped nodes

Count how many mappings landed in each Active Inference role (`OBSERVATION`, `ACTION`, `CONSTRAINT`, `POLICY`, `HIDDEN_STATE`, …). This is the answer to the central question of this notebook: *which flask patterns become which AI roles?*

In [5]:
def _role_of(mapping) -> str:
    """Return the role string for a SemanticMapping (or a dict fallback)."""
    kind = getattr(mapping, "kind", None)
    if kind is None and isinstance(mapping, dict):
        kind = mapping.get("kind")
    return getattr(kind, "value", str(kind)).upper() if kind is not None else "UNKNOWN"


role_counts = Counter(_role_of(m) for m in semantic_mappings.values())
if role_counts:
    print("Role distribution:")
    for role, count in role_counts.most_common():
        print(f"  {role:<15} {count}")
else:
    print("(no semantic mapping returned for this fixture)")

Role distribution:
  ACTION          25
  OBSERVATION     21
  HIDDEN_STATE    13
  CONTEXT         5
  POLICY          3
  CONSTRAINT      1


## 6. Which flask patterns became which role?

Group mappings by role and show a handful of the real source nodes they point at. This is where the mapping becomes intuitive:

- **OBSERVATION** — anything that *reads* a request: `get_header`, middleware `before_request`, `Request` dataclass accessors.
- **ACTION** — anything that *writes* a response: route handlers, `Response` construction, middleware `after_request`.
- **CONSTRAINT** — validation and error handlers: `ValidationError`, `errorhandler(...)`, `raise_if_*` helpers.
- **POLICY** — the dispatch loop itself: `Application.dispatch`, which selects which handler runs.

Don't be surprised if some handlers are misclassified — the rule engine is deliberately tunable (see notebook 10 for the YAML DSL).

In [6]:
from collections import defaultdict

by_role: dict[str, list[str]] = defaultdict(list)
for mapping in semantic_mappings.values():
    role = _role_of(mapping)
    fragment_ids = getattr(mapping, "graph_fragment_node_ids", None) or []
    if not fragment_ids:
        label = getattr(mapping, "semantic_label", "") or "<unlabeled>"
        by_role[role].append(label)
        continue
    for nid in fragment_ids:
        node = nodes.get(nid, {})
        qname = node.get("qualified_name") or node.get("name") or nid
        by_role[role].append(qname)

for role in sorted(by_role):
    names = sorted(set(by_role[role]))
    print(f"{role} ({len(names)}):")
    for name in names[:6]:
        print(f"    - {name}")
    if len(names) > 6:
        print(f"    ... +{len(names) - 6} more")

ACTION (25):
    - app.Application.__init__
    - app.Application._handle_error
    - app.Application.dispatch
    - app.Application.errorhandler
    - app.AuthMiddleware.__init__
    - app.LoggingMiddleware.__init__
    ... +19 more
CONSTRAINT (1):
    - models.Model.validate
CONTEXT (5):
    - config.BaseConfig
    - config.ConfigError
    - config.DevelopmentConfig
    - config.ProductionConfig
    - config.TestingConfig
HIDDEN_STATE (14):
    - app.Application
    - app.AuthMiddleware
    - app.LoggingMiddleware
    - app.Route
    - models.Field
    - models.InMemorySession
    ... +8 more
OBSERVATION (21):
    - app.Request.get_header
    - app.Response.json
    - app.Route.matches
    - config.BaseConfig.__getitem__
    - config.BaseConfig.__init__
    - config.BaseConfig._apply_environment
    ... +15 more
POLICY (3):
    - app.Application.route
    - app.Middleware
    - utils.retry


## 7. Sanity check

A Flask-style module should yield *at least* a dozen nodes and a non-empty semantic mapping. Fail loudly if either is empty — that usually means the ingest stage silently skipped the fixture.

In [7]:
assert len(nodes) >= 10, f"Expected >=10 nodes, got {len(nodes)}"
assert len(edges) >= 1, f"Expected >=1 edge, got {len(edges)}"
assert len(semantic_mappings) >= 1, f"Expected >=1 semantic mapping, got {len(semantic_mappings)}"
print(f"OK — {len(nodes)} nodes, {len(edges)} edges, {len(semantic_mappings)} semantic mappings.")

OK — 98 nodes, 154 edges, 68 semantic mappings.


## 8. Takeaways

- The forward pipeline runs on realistic framework code without any flask-specific configuration.
- Framework-level idioms map cleanly onto Active Inference roles: reads → OBSERVATION, writes → ACTION, validation → CONSTRAINT, dispatch → POLICY.
- When a node lands in the *wrong* role, you rarely need to touch Python — the YAML DSL (notebook 10) and custom plugins (notebook 09) cover almost every override you'll want.
- For bigger real-world scans, prefer `Session.export_all(...)` (see notebook 02) so the GNN markdown, state-space model, and process model are all written to disk for offline inspection.